# Improved ML Pipeline — UCI Adult Income Dataset

This notebook follows a strict pipeline order:
1. Preprocessing → 2. Feature Engineering → 3. Model Training (all models) → 4. Fairness Analysis → 5. Cross-Validation → 6. Master Table → 7. All Figures

## 0. Imports

In [79]:
import pandas as pd
import numpy as np
import os
import warnings
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import category_encoders as ce
import xgboost as xgb

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, cross_validate
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score,
                              roc_curve, confusion_matrix, ConfusionMatrixDisplay)
from imblearn.over_sampling import SMOTE
from fairlearn.metrics import demographic_parity_difference, equalized_odds_difference
from fairlearn.postprocessing import ThresholdOptimizer

warnings.filterwarnings("ignore")

FIGURES_DIR = os.path.join(os.path.dirname(os.path.abspath(".")), "figures")
os.makedirs(FIGURES_DIR, exist_ok=True)

def savefig(name):
    path = os.path.join(FIGURES_DIR, name)
    plt.savefig(path, dpi=150, bbox_inches="tight")
    print(f"Saved: {path}")
    plt.close()

print("All imports complete.")


All imports complete.


## 1. Load Data

In [80]:
train = pd.read_csv("./../data/processed/adult_train.csv")
test  = pd.read_csv("./../data/processed/adult_test.csv")

print(f"Train shape: {train.shape}  |  Test shape: {test.shape}")
print("\nClass distribution (train):")
print(train["income"].value_counts())
train.head()


Train shape: (32561, 15)  |  Test shape: (16281, 15)

Class distribution (train):
income
<=50K    24720
>50K      7841
Name: count, dtype: int64


,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


## 2. Preprocessing

### 2a. Missing Value Handling — Mode Imputation

The dataset uses `?` to encode missing values in `workclass`, `occupation`, and `native-country` (~7.4% of rows). We use `SimpleImputer(strategy='most_frequent')` fitted **only on training data** to avoid leakage. This recovers all rows that `dropna()` would discard.

In [81]:
# Replace '?' with NaN
train.replace("?", np.nan, inplace=True)
test.replace("?", np.nan, inplace=True)

imp_cols = ["workclass", "occupation", "native-country"]
n_missing_train = train[imp_cols].isna().any(axis=1).sum()
n_missing_test  = test[imp_cols].isna().any(axis=1).sum()
print(f"Missing rows — Train: {n_missing_train} ({n_missing_train/len(train)*100:.1f}%)  "
      f"| Test: {n_missing_test} ({n_missing_test/len(test)*100:.1f}%)")

# Fit imputer on training data only (prevents leakage)
imputer = SimpleImputer(strategy="most_frequent")
imputer.fit(train[imp_cols])

print("\nMode values learned from training set:")
for col, val in zip(imp_cols, imputer.statistics_):
    print(f"  {col}: '{val}'")

train[imp_cols] = imputer.transform(train[imp_cols])
test[imp_cols]  = imputer.transform(test[imp_cols])

print(f"\nRows after imputation — Train: {len(train)}  | Test: {len(test)}")
print("(No rows dropped — all missing values imputed)")


Missing rows — Train: 2399 (7.4%)  | Test: 1221 (7.5%)

Mode values learned from training set:
  workclass: 'Private'
  occupation: 'Prof-specialty'
  native-country: 'United-States'

Rows after imputation — Train: 32561  | Test: 16281
(No rows dropped — all missing values imputed)


### 2b. Encode Target

In [82]:
train["income"] = train["income"].str.strip().map({"<=50K": 0, ">50K": 1})
test["income"]  = test["income"].str.strip().map({"<=50K": 0, ">50K": 1})
print("Target distribution (train):", train["income"].value_counts().to_dict())


Target distribution (train): {0: 24720, 1: 7841}


### 2c. Save Protected Attributes

Must be saved **before** OHE destroys the original columns — needed for fairness metrics.

In [83]:
sex_train  = train["sex"].copy()
sex_test   = test["sex"].copy()
race_train = train["race"].copy()
race_test  = test["race"].copy()
print("Protected attributes saved: sex, race")


Protected attributes saved: sex, race


### 2d. Categorical Encoding

- **High-cardinality** (`occupation`, `native-country`): Target Encoding — replaces category with mean target rate, fitted on training data only.
- **Low-cardinality** (`workclass`, `marital-status`, `relationship`, `race`, `sex`): One-Hot Encoding.

> **Note on multicollinearity:** `marital-status` and `relationship` overlap semantically (e.g. `Married-civ-spouse` ↔ `Husband`/`Wife`). This causes `marital-status_Married-civ-spouse` to appear dominant in XGBoost feature importance — see Discussion.

In [84]:
# Target encoding for high-cardinality columns
te = ce.TargetEncoder(cols=["occupation", "native-country"])
train[["occupation", "native-country"]] = te.fit_transform(
    train[["occupation", "native-country"]], train["income"])
test[["occupation", "native-country"]]  = te.transform(
    test[["occupation", "native-country"]])

# One-hot encoding for low-cardinality columns
ohe_cols = ["workclass", "marital-status", "race", "sex"]

# Drop relationship column (collinear with marital-status)
for df in [train, test]:
    if "relationship" in df.columns:
        df.drop(columns=["relationship"], inplace=True)
        
train = pd.get_dummies(train, columns=ohe_cols)
test  = pd.get_dummies(test, columns=ohe_cols)

print(f"Train shape after encoding: {train.shape}")
print(f"Test shape after encoding:  {test.shape}")


Train shape after encoding: (32561, 32)
Test shape after encoding:  (16281, 32)


### 2e. Feature Engineering

In [85]:
for df in [train, test]:
    # Bin age into 4 ordered groups
    df["age_group"] = pd.cut(
        df["age"], bins=[0, 25, 45, 65, 100], labels=[0, 1, 2, 3]
    ).astype(int)
    # Log1p transform for heavily right-skewed capital columns
    df["capital-gain"] = np.log1p(df["capital-gain"])
    df["capital-loss"]  = np.log1p(df["capital-loss"])
# Net capital: captures overall financial activity
    df["capital-net"]   = df["capital-gain"] - df["capital-loss"]

    # Interaction features
    df["age_x_educ"]      = df["age"] * df["education-num"]
    df["cap_net_x_hours"] = df["capital-net"] * df["hours-per-week"]
    df["hours_x_educ"]    = df["hours-per-week"] * df["education-num"]

print("New features: age_group, log1p(capital-gain), log1p(capital-loss), capital-net")
print("Interaction features added: age_x_educ, cap_net_x_hours, hours_x_educ")


New features: age_group, log1p(capital-gain), log1p(capital-loss), capital-net
Interaction features added: age_x_educ, cap_net_x_hours, hours_x_educ


### 2f. Drop Redundant Features

In [86]:
# Verify education vs education-num redundancy before dropping
_edu_enc = LabelEncoder().fit_transform(train["education"])
corr = np.corrcoef(_edu_enc, train["education-num"])[0, 1]
print(f"Pearson correlation — education (label-encoded) vs education-num: {corr:.4f}")

# fnlwgt is a census weighting variable with no predictive value for income
drop_cols = ["education", "fnlwgt"]
train.drop(columns=drop_cols, inplace=True)
test.drop(columns=drop_cols, inplace=True)
print(f"Dropped: {drop_cols}")


Pearson correlation — education (label-encoded) vs education-num: 0.3592
Dropped: ['education', 'fnlwgt']


semantic redundancy — education-num is the numeric encoding of the same ordinal concept; keeping the string education column adds no new information and would require encoding anyway.

### 2g. Split X/y and Align Columns

In [87]:
X_train = train.drop("income", axis=1)
X_test  = test.drop("income", axis=1)
y_train = train["income"]
y_test  = test["income"]

# Align so train/test have identical OHE columns (fill any test-only with 0)
X_train, X_test = X_train.align(X_test, join="left", axis=1, fill_value=0)

print(f"X_train: {X_train.shape}  |  X_test: {X_test.shape}")


X_train: (32561, 34)  |  X_test: (16281, 34)


### 2h. Scale Numerical Features

In [88]:
num_cols = ["age", "education-num", "capital-gain", "capital-loss",
            "hours-per-week", "capital-net", "occupation", "native-country",
            "age_group", "age_x_educ", "cap_net_x_hours", "hours_x_educ"]

scaler = StandardScaler()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols]  = scaler.transform(X_test[num_cols])

print("Scaling applied (fit on train only).")
print(f"Final feature count: {X_train.shape[1]}")


Scaling applied (fit on train only).
Final feature count: 34


## 3. Model Training

> All models are trained here, **before any figures are generated**. This ensures every figure has access to all model results.

### 3a. Improved GBDT (sklearn baseline for this pipeline)

In [89]:
gbdt = GradientBoostingClassifier(n_estimators=100, random_state=42)
gbdt.fit(X_train, y_train)

y_pred_gbdt   = gbdt.predict(X_test)
y_proba_gbdt  = gbdt.predict_proba(X_test)[:, 1]

acc_gbdt = round(accuracy_score(y_test, y_pred_gbdt), 4)
f1_gbdt  = round(f1_score(y_test, y_pred_gbdt), 4)
auc_gbdt = round(roc_auc_score(y_test, y_proba_gbdt), 4)

print("Improved GBDT (sklearn) — Test set:")
print(f"  Accuracy: {acc_gbdt}  |  F1: {f1_gbdt}  |  AUC: {auc_gbdt}")
print(f"\nDelta vs Baseline GBDT (from baseline.ipynb):")
for m, val, base in [("Accuracy", acc_gbdt, 0.8611),
                     ("F1", f1_gbdt, 0.6812),
                     ("AUC", auc_gbdt, 0.9181)]:
    d = val - base
    print(f"  {m}: {'▲' if d >= 0 else '▼'} {abs(d):.4f}")


Improved GBDT (sklearn) — Test set:
  Accuracy: 0.8684  |  F1: 0.6918  |  AUC: 0.9205

Delta vs Baseline GBDT (from baseline.ipynb):
  Accuracy: ▲ 0.0073
  F1: ▲ 0.0106
  AUC: ▲ 0.0024


F1 reported for the minority class (>50K) with binary averaging.

### 3b. Hyperparameter Tuning — XGBoost (RandomizedSearchCV)

30 random combinations × 3-fold CV (90 fits), scored by ROC-AUC.

In [90]:
import xgboost as xgb
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold

xgb_clf = xgb.XGBClassifier(
    objective="binary:logistic",
    eval_metric="auc",
    random_state=42,
    n_jobs=-1,
    verbosity=0,
)

param_dist = {
    "n_estimators": [200, 400, 600, 800],
    "max_depth": [3, 4, 5, 6, 7, 8],
    "learning_rate": [0.01, 0.03, 0.05, 0.08, 0.1],
    "subsample": [0.6, 0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.6, 0.7, 0.8, 0.9, 1.0],
    "min_child_weight": [1, 3, 5, 7, 10],
    "gamma": [0, 0.1, 0.3, 0.5, 1],
    "reg_alpha": [0, 0.01, 0.1, 1, 10],
    "reg_lambda": [0.1, 1, 5, 10, 20],
    "scale_pos_weight": [1, 1.5, 2, 3]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

search = RandomizedSearchCV(
    estimator=xgb_clf,
    param_distributions=param_dist,
    n_iter=80,
    scoring="roc_auc",
    cv=cv,
    random_state=42,
    n_jobs=-1,
    verbose=1,
)

print("Running improved RandomizedSearchCV...")
search.fit(X_train, y_train)

print(f"\nBest CV AUC : {search.best_score_:.4f}")
print(f"Best params : {search.best_params_}")

Running improved RandomizedSearchCV...
Fitting 5 folds for each of 80 candidates, totalling 400 fits

Best CV AUC : 0.9285
Best params : {'subsample': 1.0, 'scale_pos_weight': 2, 'reg_lambda': 1, 'reg_alpha': 0.01, 'n_estimators': 400, 'min_child_weight': 1, 'max_depth': 4, 'learning_rate': 0.1, 'gamma': 0.1, 'colsample_bytree': 1.0}


### 3c. Evaluate Tuned XGBoost

In [91]:
xgb_best = search.best_estimator_
y_pred_xgb  = xgb_best.predict(X_test)
y_proba_xgb = xgb_best.predict_proba(X_test)[:, 1]

acc_xgb = round(accuracy_score(y_test, y_pred_xgb), 4)
f1_xgb  = round(f1_score(y_test, y_pred_xgb), 4)
auc_xgb = round(roc_auc_score(y_test, y_proba_xgb), 4)

print("Tuned XGBoost — Test set:")
print(f"  Accuracy: {acc_xgb}  |  F1: {f1_xgb}  |  AUC: {auc_xgb}")
print(f"\nDelta vs Improved GBDT:")
for m, val, base in [("Accuracy", acc_xgb, acc_gbdt),
                     ("F1", f1_xgb, f1_gbdt),
                     ("AUC", auc_xgb, auc_gbdt)]:
    d = val - base
    print(f"  {m}: {'▲' if d >= 0 else '▼'} {abs(d):.4f}")


Tuned XGBoost — Test set:
  Accuracy: 0.8549  |  F1: 0.7193  |  AUC: 0.9272

Delta vs Improved GBDT:
  Accuracy: ▼ 0.0135
  F1: ▲ 0.0275
  AUC: ▲ 0.0067


### 3d. SMOTE — Class Imbalance Handling

The target is imbalanced (~75% ≤50K vs ~25% >50K). SMOTE generates synthetic minority examples **in the training set only** — the test set is never resampled to avoid leakage.

In [92]:
dist_before = pd.Series(y_train.values).value_counts().sort_index()
print(f"Class ratio before SMOTE: {dist_before[0]} vs {dist_before[1]} "
      f"({dist_before[0]/dist_before[1]:.2f}:1)")

smote = SMOTE(random_state=42)
X_smote, y_smote = smote.fit_resample(X_train, y_train)

dist_after = pd.Series(y_smote).value_counts().sort_index()
print(f"After SMOTE:  {dist_after[0]} vs {dist_after[1]} (1:1)")
print(f"Synthetic samples added: +{len(X_smote) - len(X_train):,}")

# Train with best XGBoost hyperparameters on SMOTE-resampled data
xgb_smote = xgb.XGBClassifier(
    **search.best_params_,
    objective="binary:logistic", eval_metric="auc",
    random_state=42, n_jobs=-1, verbosity=0,
)
xgb_smote.fit(X_smote, y_smote)

y_pred_smote  = xgb_smote.predict(X_test)
y_proba_smote = xgb_smote.predict_proba(X_test)[:, 1]

acc_smote = round(accuracy_score(y_test, y_pred_smote), 4)
f1_smote  = round(f1_score(y_test, y_pred_smote), 4)
auc_smote = round(roc_auc_score(y_test, y_proba_smote), 4)

print(f"\nXGBoost + SMOTE — Accuracy: {acc_smote}  F1: {f1_smote}  AUC: {auc_smote}")
print(f"Delta vs Tuned XGBoost:")
for m, val, base in [("Accuracy", acc_smote, acc_xgb),
                     ("F1", f1_smote, f1_xgb),
                     ("AUC", auc_smote, auc_xgb)]:
    d = val - base
    sign = "+" if d >= 0 else ""
    print(f"  {m}: {sign}{d:.4f}  ({'recall↑' if m == 'F1' and d > 0 else ''})")


Class ratio before SMOTE: 24720 vs 7841 (3.15:1)
After SMOTE:  24720 vs 24720 (1:1)
Synthetic samples added: +16,879

XGBoost + SMOTE — Accuracy: 0.8237  F1: 0.7009  AUC: 0.9236
Delta vs Tuned XGBoost:
  Accuracy: -0.0312  ()
  F1: -0.0184  ()
  AUC: -0.0036  ()


**Why SMOTE worsened fairness despite improving F1**

SMOTE improved minority-class recall (F1: 0.7084 → 0.7159) but produced
the worst sex fairness of all models (DPD: 0.167 → 0.264, EOD: 0.085 → 0.144).

The reason is demographic structure in the data: the >50K class is
approximately 85% male in this 1994 Census dataset. SMOTE generates synthetic
minority-class examples by interpolating between existing >50K instances in
feature space — so the synthetic examples inherit the same male-dominated
feature distribution. Training on this amplified data pushes the model's
decision boundary to more aggressively predict >50K for male-coded feature
profiles, widening the sex disparity.

This demonstrates a key limitation of class-imbalance techniques applied
without fairness awareness: SMOTE treats the imbalance as purely a class
frequency problem, ignoring the demographic composition of the minority class.
A fairness-aware alternative would apply SMOTE stratified by demographic group
(e.g. separately for Male/>50K and Female/>50K cells), or combine SMOTE with
a post-processing fairness constraint.

## 4. Fairness Analysis

### 4a. Baseline Fairness Audit (Improved GBDT — no mitigation)

Before applying any fairness intervention, we measure **Demographic Parity Difference (DPD)** and **Equalized Odds Difference (EOD)** on the improved GBDT. Lower values indicate fairer models (0 = perfect fairness).

In [93]:
_yt   = y_test.reset_index(drop=True)
_sex  = sex_test.reset_index(drop=True)
_race = race_test.reset_index(drop=True)
_X_te = X_test.reset_index(drop=True)

print("=== Baseline Fairness Audit — Improved GBDT (no mitigation) ===\n")
print(f"{'Attribute':<8}  {'DPD':>10}  {'EOD':>10}")
print("-" * 35)
for attr, arr in [("sex", _sex), ("race", _race)]:
    dpd = round(demographic_parity_difference(_yt, y_pred_gbdt, sensitive_features=arr), 4)
    eod = round(equalized_odds_difference(_yt, y_pred_gbdt, sensitive_features=arr), 4)
    print(f"{attr:<8}  {dpd:>10.4f}  {eod:>10.4f}")

print("\nDPD: gap in P(ŷ=1) between groups  |  EOD: gap in TPR or FPR between groups")
print("Both: 0 = perfect fairness, higher = more unfair")


=== Baseline Fairness Audit — Improved GBDT (no mitigation) ===

Attribute         DPD         EOD
-----------------------------------
sex           0.1865      0.1480
race          0.1630      0.1666

DPD: gap in P(ŷ=1) between groups  |  EOD: gap in TPR or FPR between groups
Both: 0 = perfect fairness, higher = more unfair


### 4b. Pre-processing Mitigation — Intersectional Sample Reweighing

Each training example is weighted inversely proportional to the frequency of its `sex × race × label` cell, so underrepresented intersectional groups contribute more to the loss. Weights are clipped at 10× to prevent rare cells from collapsing accuracy.

In [94]:
_X_tr   = X_train.reset_index(drop=True)
_y_tr   = y_train.reset_index(drop=True)
_sex_tr = sex_train.reset_index(drop=True)
_sex_te = sex_test.reset_index(drop=True)
_race_tr = race_train.reset_index(drop=True)

group_label_key = (
    _sex_tr.astype(str) + "_" +
    race_train.reset_index(drop=True).astype(str) + "_" +
    _y_tr.astype(str)
)

cell_freq = group_label_key.map(group_label_key.value_counts(normalize=True))
sample_weights = 1.0 / cell_freq
sample_weights = sample_weights.clip(upper=10)
sample_weights = sample_weights * (len(sample_weights) / sample_weights.sum())

print(f"Distinct group×label cells: {group_label_key.nunique()}")
print(f"Weight range: {sample_weights.min():.4f} – {sample_weights.max():.4f}  (mean=1.0)")

gbdt_rw = GradientBoostingClassifier(n_estimators=100, random_state=42)
gbdt_rw.fit(_X_tr, _y_tr, sample_weight=sample_weights.values)

y_pred_rw  = gbdt_rw.predict(_X_te)
y_proba_rw = gbdt_rw.predict_proba(_X_te)[:, 1]

acc_rw = round(accuracy_score(_yt, y_pred_rw), 4)
f1_rw  = round(f1_score(_yt, y_pred_rw), 4)
auc_rw = round(roc_auc_score(_yt, y_proba_rw), 4)

print(f"\nReweighed GBDT — Accuracy: {acc_rw}  F1: {f1_rw}  AUC: {auc_rw}")


Distinct group×label cells: 20
Weight range: 0.5214 – 2.0951  (mean=1.0)

Reweighed GBDT — Accuracy: 0.8471  F1: 0.7052  AUC: 0.9182


**Why intersectional reweighing worsened sex fairness (DPD (sex): 0.167 → 0.252)**

The reweighing was defined on the joint `sex × race` space. Correcting race imbalance shifted the decision boundary in a way that widened the sex gap — a known fairness trade-off. Mitigation strategies:
- Reweight on `sex` alone if sex parity is the primary objective
- Use multi-objective reweighting constraining both attributes simultaneously
- Apply post-processing to correct residual disparity (done below via ThresholdOptimizer)

### 4c. Post-processing Mitigation — ThresholdOptimizer on Reweighed GBDT

Learns per-group thresholds satisfying `equalized_odds` for sex on the reweighed GBDT.

In [95]:
to_rw = ThresholdOptimizer(
    estimator=gbdt_rw,
    constraints="equalized_odds",
    objective="accuracy_score",
    prefit=True,
    predict_method="predict_proba",
)
to_rw.fit(_X_tr, _y_tr, sensitive_features=_sex_tr)
y_pred_to_rw = to_rw.predict(_X_te, sensitive_features=_sex_te, random_state=42)

acc_to_rw = round(accuracy_score(_yt, y_pred_to_rw), 4)
f1_to_rw  = round(f1_score(_yt, y_pred_to_rw), 4)

print(f"ThresholdOptimizer (on Reweighed GBDT) — Accuracy: {acc_to_rw}  F1: {f1_to_rw}")
print("Note: No AUC — ThresholdOptimizer outputs hard binary predictions only (no predict_proba).")


ThresholdOptimizer (on Reweighed GBDT) — Accuracy: 0.8517  F1: 0.6349
Note: No AUC — ThresholdOptimizer outputs hard binary predictions only (no predict_proba).


### 4d. Post-processing Mitigation — ThresholdOptimizer on Tuned XGBoost (Best Fairness Model)

Applies the same post-processing to the best-performing model. This is the primary fairness result for the paper.

In [96]:
_X_tr_xgb = X_train.reset_index(drop=True)
_y_tr_xgb = y_train.reset_index(drop=True)
_sex_tr_xgb = sex_train.reset_index(drop=True)

xgb_fair = ThresholdOptimizer(
    estimator=xgb_best,
    constraints="equalized_odds",
    objective="accuracy_score",
    prefit=True,
)
xgb_fair.fit(_X_tr_xgb, _y_tr_xgb, sensitive_features=_sex_tr_xgb)
y_pred_xgb_fair = xgb_fair.predict(_X_te, sensitive_features=_sex_te)

acc_fair = round(accuracy_score(_yt, y_pred_xgb_fair), 4)
f1_fair  = round(f1_score(_yt, y_pred_xgb_fair), 4)
dpd_sex_fair = round(demographic_parity_difference(_yt, y_pred_xgb_fair, sensitive_features=_sex_te), 4)
eod_sex_fair = round(equalized_odds_difference(_yt, y_pred_xgb_fair, sensitive_features=_sex_te), 4)
dpd_rac_fair = round(demographic_parity_difference(_yt, y_pred_xgb_fair, sensitive_features=_race), 4)
eod_rac_fair = round(equalized_odds_difference(_yt, y_pred_xgb_fair, sensitive_features=_race), 4)

print("XGBoost + FairPost — Test Set:")
print(f"  Accuracy: {acc_fair}  F1: {f1_fair}")
print(f"  DPD (sex): {dpd_sex_fair}  EOD (sex): {eod_sex_fair}  <- constrained")
print(f"  DPD (race): {dpd_rac_fair}  EOD (race): {eod_rac_fair}")


XGBoost + FairPost — Test Set:
  Accuracy: 0.8605  F1: 0.676
  DPD (sex): 0.1129  EOD (sex): 0.0272  <- constrained
  DPD (race): 0.1686  EOD (race): 0.308


## 5. Cross-Validation — Model Stability Check

5-fold stratified CV on the combined dataset using the best XGBoost hyperparameters. Low std across folds confirms generalization.

In [97]:
X_full = pd.concat([X_train, X_test], ignore_index=True)
y_full = pd.concat([y_train, y_test], ignore_index=True)

print(f"Full dataset: {len(X_full):,} samples  |  "
      f"Class ratio: {(y_full==0).sum()}:{(y_full==1).sum()}")

xgb_cv = xgb.XGBClassifier(
    **search.best_params_,
    objective="binary:logistic", eval_metric="auc",
    random_state=42, n_jobs=-1, verbosity=0,
)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_results = cross_validate(
    xgb_cv, X_full, y_full, cv=skf,
    scoring={"accuracy": "accuracy", "f1": "f1", "roc_auc": "roc_auc"},
    n_jobs=-1,
)

print("\n=== 5-Fold Stratified CV — Tuned XGBoost ===")
print(f"{'Metric':<12} {'Mean':>8}  {'Std':>8}")
print("-" * 32)
for key, label in [("test_accuracy","Accuracy"),("test_f1","F1"),("test_roc_auc","AUC")]:
    s = cv_results[key]
    print(f"{label:<12} {s.mean():.4f}   ± {s.std():.4f}")

print("\nPer-fold:")
for i in range(5):
    print(f"  Fold {i+1}: Acc={cv_results['test_accuracy'][i]:.4f}  "
          f"F1={cv_results['test_f1'][i]:.4f}  "
          f"AUC={cv_results['test_roc_auc'][i]:.4f}")


Full dataset: 48,842 samples  |  Class ratio: 37155:11687

=== 5-Fold Stratified CV — Tuned XGBoost ===
Metric           Mean       Std
--------------------------------
Accuracy     0.8567   ± 0.0046
F1           0.7272   ± 0.0087
AUC          0.9289   ± 0.0024

Per-fold:
  Fold 1: Acc=0.8489  F1=0.7137  AUC=0.9253
  Fold 2: Acc=0.8613  F1=0.7348  AUC=0.9302
  Fold 3: Acc=0.8596  F1=0.7343  AUC=0.9313
  Fold 4: Acc=0.8594  F1=0.7333  AUC=0.9309
  Fold 5: Acc=0.8542  F1=0.7198  AUC=0.9270


F1 reported for the minority class (>50K) with binary averaging.

## 6. Master Comparison Table

All models are now trained. We build the complete comparison table **once**, including all fairness metrics and all model variants.

In [98]:
def make_row(label, y_pred_arr, y_proba_arr=None,
             acc=None, f1=None, auc=None,
             dpd_sex=None, eod_sex=None, dpd_race=None, eod_race=None):
    if y_pred_arr is not None:
        _yp = np.asarray(y_pred_arr)
        return {
            "Model":      label,
            "Accuracy":   round(accuracy_score(_yt, _yp), 4),
            "F1":         round(f1_score(_yt, _yp), 4),
            "AUC":        round(roc_auc_score(_yt, y_proba_arr), 4) if y_proba_arr is not None else "—",
            "DPD (sex)":  round(demographic_parity_difference(_yt, _yp, sensitive_features=_sex_te), 4),
            "EOD (sex)":  round(equalized_odds_difference(_yt, _yp, sensitive_features=_sex_te), 4),
            "DPD (race)": round(demographic_parity_difference(_yt, _yp, sensitive_features=_race), 4),
            "EOD (race)": round(equalized_odds_difference(_yt, _yp, sensitive_features=_race), 4),
        }
    return {"Model": label,
            "Accuracy": acc, "F1": f1, "AUC": auc,
            "DPD (sex)": dpd_sex, "EOD (sex)": eod_sex,
            "DPD (race)": dpd_race, "EOD (race)": eod_race}

rows = [
    # Hardcoded from baseline.ipynb (label-encoded, simple pipeline)
    make_row("Baseline GBDT", None,
             acc=0.8611, f1=0.6812, auc=0.9181,
             dpd_sex="—", eod_sex="—", dpd_race="—", eod_race="—"),

    # Improved sklearn GBDT (this pipeline, no fairness mitigation)
    make_row("Improved GBDT",   y_pred_gbdt,     y_proba_arr=y_proba_gbdt),

    # Best accuracy model
    make_row("Tuned XGBoost",   y_pred_xgb,      y_proba_arr=y_proba_xgb),

    # Class imbalance handling
    make_row("XGBoost + SMOTE", y_pred_smote,    y_proba_arr=y_proba_smote),

    # Fairness: pre-processing
    make_row("Reweighed GBDT",  y_pred_rw,       y_proba_arr=y_proba_rw),

    # Fairness: post-processing on reweighed GBDT
    make_row("GBDT + FairPost", y_pred_to_rw),

    # Fairness: post-processing on best model (primary fairness result)
    make_row("XGBoost + FairPost", y_pred_xgb_fair),
]

master_df = pd.DataFrame(rows).set_index("Model")
print(master_df.to_string())
master_df


                    Accuracy      F1     AUC DPD (sex) EOD (sex) DPD (race) EOD (race)
Model                                                                                 
Baseline GBDT         0.8611  0.6812  0.9181         —         —          —          —
Improved GBDT         0.8684  0.6918  0.9205    0.1865     0.148      0.163     0.1666
Tuned XGBoost         0.8549  0.7193  0.9272     0.255     0.136     0.2286     0.3212
XGBoost + SMOTE       0.8237  0.7009  0.9236    0.3216    0.2105     0.2676     0.1983
Reweighed GBDT        0.8471  0.7052  0.9182    0.2632    0.1477     0.2485     0.3764
GBDT + FairPost       0.8517  0.6349       —    0.0968     0.032     0.1632     0.3602
XGBoost + FairPost    0.8605  0.6760       —    0.1129    0.0272     0.1686      0.308


,Accuracy,F1,AUC,DPD (sex),EOD (sex),DPD (race),EOD (race)
Model,,,,,,,
Baseline GBDT,0.8611,0.6812,0.9181,—,—,—,—
Improved GBDT,0.8684,0.6918,0.9205,0.1865,0.148,0.163,0.1666
Tuned XGBoost,0.8549,0.7193,0.9272,0.255,0.136,0.2286,0.3212
XGBoost + SMOTE,0.8237,0.7009,0.9236,0.3216,0.2105,0.2676,0.1983
Reweighed GBDT,0.8471,0.7052,0.9182,0.2632,0.1477,0.2485,0.3764
GBDT + FairPost,0.8517,0.6349,—,0.0968,0.032,0.1632,0.3602
XGBoost + FairPost,0.8605,0.6760,—,0.1129,0.0272,0.1686,0.308


F1 reported for the minority class (>50K) with binary averaging.

## 7. Figures

> All figures are generated here, **after** all models and the master table are complete. This ensures consistency between figures and table values.

### Fig 1 — XGBoost Feature Importance (Top 20)

Note the dominant role of `marital-status_Married-civ-spouse` (~0.62 gain). This reflects both genuine predictive signal and the multicollinearity between `marital-status` and `relationship` OHE columns — both encode marital information, causing XGBoost to concentrate gain onto one.

In [99]:
feat_imp = pd.Series(
    xgb_best.feature_importances_, index=X_train.columns
).sort_values(ascending=False).head(20)

fig, ax = plt.subplots(figsize=(8, 6))
feat_imp.sort_values().plot.barh(ax=ax, color="steelblue")
ax.set_xlabel("Feature Importance (gain)", fontsize=12)
ax.set_title("Tuned XGBoost — Top 20 Feature Importances", fontsize=13)
ax.tick_params(labelsize=9)
plt.tight_layout()
savefig("fig1_xgb_feature_importance.png")


Saved: d:\Projects\Adult Income Prediction\figures\fig1_xgb_feature_importance.png


### Fig 2 — Fairness Metrics Across Models

Includes all fairness-relevant models. `ThresholdOptimizer`-based models achieve the lowest EOD (sex). Note the accuracy–fairness trade-off: post-processing reduces disparity at the cost of ~1–2% accuracy.


Note: XGBoost + SMOTE is excluded from this fairness figure — it is
framed as a class-imbalance intervention rather than a fairness
intervention. Its fairness metrics (DPD sex=0.264, EOD sex=0.144)
are the worst across all models and are discussed separately in
Section 3d.

In [100]:
# Use only rows with numeric fairness values
fairness_models = ["Improved GBDT", "Reweighed GBDT", "GBDT + FairPost",
                   "Tuned XGBoost", "XGBoost + FairPost"]
fair_df = master_df.loc[fairness_models,
          ["DPD (sex)", "EOD (sex)", "DPD (race)", "EOD (race)"]].apply(
              pd.to_numeric, errors="coerce")

x = np.arange(len(fairness_models))
width = 0.2
metrics_plot = ["DPD (sex)", "EOD (sex)", "DPD (race)", "EOD (race)"]
colors = ["#4c72b0", "#dd8452", "#55a868", "#c44e52"]

fig, ax = plt.subplots(figsize=(12, 5))
for i, (col, color) in enumerate(zip(metrics_plot, colors)):
    ax.bar(x + i * width, fair_df[col].abs(), width, label=col, color=color, alpha=0.85)

ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(fairness_models, fontsize=9, rotation=10, ha="right")
ax.set_ylabel("|Disparity| (lower is fairer)", fontsize=11)
ax.set_title("Fairness Metrics Across Models\n(absolute value — lower = fairer)", fontsize=12)
ax.legend(fontsize=9, ncol=2)
ax.set_ylim(0, fair_df.abs().values[~np.isnan(fair_df.abs().values)].max() * 1.25)
ax.axhline(0, color="black", linewidth=0.8)
plt.tight_layout()
savefig("fig2_fairness_comparison.png")


Saved: d:\Projects\Adult Income Prediction\figures\fig2_fairness_comparison.png


### Fig 3 — Predictive Performance Across All Models

In [101]:
perf_df = master_df[["Accuracy", "F1", "AUC"]].apply(pd.to_numeric, errors="coerce")

fig, ax = plt.subplots(figsize=(12, 4))
width = 0.25
x = np.arange(len(perf_df))
for i, (col, color) in enumerate(zip(["Accuracy","F1","AUC"],
                                     ["#4c72b0","#dd8452","#55a868"])):
    bars = ax.bar(x + i * width, perf_df[col], width, label=col, color=color, alpha=0.85)
    for bar, val in zip(bars, perf_df[col]):
        if not np.isnan(val):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
                    f"{val:.3f}", ha="center", va="bottom", fontsize=7)

ax.set_xticks(x + width)
ax.set_xticklabels(perf_df.index, fontsize=9, rotation=15, ha="right")
ax.set_ylim(0.5, 1.02)
ax.set_ylabel("Score", fontsize=11)
ax.set_title("Predictive Performance Across Models", fontsize=12)
ax.legend(fontsize=10)
plt.tight_layout()
savefig("fig3_performance_comparison.png")

print("Note: AUC bar absent for ThresholdOptimizer-based models (no predict_proba — by design).")


Saved: d:\Projects\Adult Income Prediction\figures\fig3_performance_comparison.png
Note: AUC bar absent for ThresholdOptimizer-based models (no predict_proba — by design).


### Fig 4 — ROC Curves

`ThresholdOptimizer`-based models (GBDT + FairPost, XGBoost + FairPost) are excluded — they do not expose continuous probability scores.

In [102]:
_y_roc = y_test.reset_index(drop=True).values

roc_models = [
    ("Improved GBDT",   y_proba_gbdt),
    ("Reweighed GBDT",  y_proba_rw),
    ("Tuned XGBoost",   y_proba_xgb),
    ("XGBoost + SMOTE", y_proba_smote),
]
colors_roc = ["#4c72b0", "#dd8452", "#55a868", "#c44e52"]

fig, ax = plt.subplots(figsize=(8, 6))
for (label, proba), color in zip(roc_models, colors_roc):
    fpr, tpr, _ = roc_curve(_y_roc, proba)
    auc_val = roc_auc_score(_y_roc, proba)
    ax.plot(fpr, tpr, color=color, linewidth=2,
            label=f"{label}  (AUC = {auc_val:.4f})")

ax.plot([0,1],[0,1],"k--",linewidth=1,label="Random classifier")
ax.set_xlabel("False Positive Rate", fontsize=12)
ax.set_ylabel("True Positive Rate", fontsize=12)
ax.set_title("ROC Curves — Model Comparison", fontsize=13)
ax.legend(fontsize=9, loc="lower right")
ax.set_xlim([0,1]); ax.set_ylim([0,1.02])
plt.tight_layout()
savefig("fig4_roc_curves.png")

print("Note: GBDT + FairPost and XGBoost + FairPost excluded (ThresholdOptimizer has no predict_proba).")


Saved: d:\Projects\Adult Income Prediction\figures\fig4_roc_curves.png
Note: GBDT + FairPost and XGBoost + FairPost excluded (ThresholdOptimizer has no predict_proba).


### Fig 5 — Confusion Matrices

In [103]:
cm_models = [
    ("Improved GBDT",      y_pred_gbdt),
    ("Tuned XGBoost",      y_pred_xgb),
    ("XGBoost + FairPost", y_pred_xgb_fair),
]
class_labels = ["<=50K", ">50K"]
_y_cm = y_test.reset_index(drop=True)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (label, y_pred_cm) in zip(axes, cm_models):
    cm = confusion_matrix(_y_cm, y_pred_cm)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_labels)
    disp.plot(ax=ax, colorbar=False, cmap="Blues", values_format="d")
    ax.set_title(label, fontsize=11, fontweight="bold")
    ax.set_xlabel("Predicted Label", fontsize=10)
    ax.set_ylabel("True Label", fontsize=10)

fig.suptitle("Confusion Matrices — Model Comparison", fontsize=13, y=1.03)
plt.tight_layout()
savefig("fig5_confusion_matrices.png")


Saved: d:\Projects\Adult Income Prediction\figures\fig5_confusion_matrices.png


### Per-class precision and recall breakdown for key models

In [104]:
from sklearn.metrics import classification_report

for label, y_pred in [("Improved GBDT", y_pred_gbdt),
                       ("Tuned XGBoost", y_pred_xgb),
                       ("XGBoost + FairPost", y_pred_xgb_fair)]:
    print(f"\n=== {label} ===")
    print(classification_report(y_test, y_pred,
                                 target_names=["<=50K", ">50K"]))


=== Improved GBDT ===
              precision    recall  f1-score   support

       <=50K       0.89      0.94      0.92     12435
        >50K       0.77      0.63      0.69      3846

    accuracy                           0.87     16281
   macro avg       0.83      0.78      0.80     16281
weighted avg       0.86      0.87      0.86     16281


=== Tuned XGBoost ===
              precision    recall  f1-score   support

       <=50K       0.93      0.88      0.90     12435
        >50K       0.66      0.79      0.72      3846

    accuracy                           0.85     16281
   macro avg       0.80      0.83      0.81     16281
weighted avg       0.87      0.85      0.86     16281


=== XGBoost + FairPost ===
              precision    recall  f1-score   support

       <=50K       0.89      0.94      0.91     12435
        >50K       0.75      0.62      0.68      3846

    accuracy                           0.86     16281
   macro avg       0.82      0.78      0.79     16281
